# Fraud Detection – Feature Reduction

Drops constant/correlated features, selects top 50 via MI+RF consensus, and applies PCA for a third feature set.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)


## 2. Load Preprocessed Data

All features are already uniformly StandardScaled from `02_preprocessing.ipynb`.

In [ ]:
X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')

# Load as pandas Series (not numpy array) for safe index-based alignment
y_train = pd.read_parquet('../data/processed/y_train.parquet').iloc[:, 0]
y_test  = pd.read_parquet('../data/processed/y_test.parquet').iloc[:, 0]

# Align indices (parquet saved with index=False -> 0-based RangeIndex)
X_train.index = y_train.index = pd.RangeIndex(len(X_train))
X_test.index  = y_test.index  = pd.RangeIndex(len(X_test))

metadata = joblib.load('../data/processed/feature_metadata.pkl')
cat_cols  = metadata.get('cat_cols', [])

print('Training set:', X_train.shape)
print('Test set:    ', X_test.shape)
print('Fraud rate   (train): {:.3%}'.format(y_train.mean()))


## 3. Remove Constant & Highly Correlated Features

In [ ]:
# 3.1 Constant columns (zero variance)
variance_mask = X_train.var() > 0
constant_cols = X_train.columns[~variance_mask].tolist()
if constant_cols:
    print(f'Dropping {len(constant_cols)} constant columns.')
    X_train = X_train.loc[:, variance_mask]
    X_test  = X_test.loc[:, variance_mask]
else:
    print('No constant columns found.')

# 3.2 Highly correlated features (correlation > 0.98)
# Use 10% sample for speed
corr_sample = X_train.sample(frac=0.1, random_state=RANDOM_STATE).corr().abs()
upper       = corr_sample.where(np.triu(np.ones(corr_sample.shape), k=1).astype(bool))
to_drop     = [col for col in upper.columns if any(upper[col] > 0.98)]
if to_drop:
    print(f'Dropping {len(to_drop)} highly correlated features (>0.98).')
    X_train = X_train.drop(columns=to_drop)
    X_test  = X_test.drop(columns=to_drop)
else:
    print('No highly correlated features found.')

print('Shape after cleaning:', X_train.shape)

# Save full (cleaned) feature set
X_train.to_parquet('../data/processed/X_train_full.parquet', index=False)
X_test.to_parquet( '../data/processed/X_test_full.parquet',  index=False)
X_train.to_csv('../data/processed/X_train_full.csv', index=False)
X_test.to_csv( '../data/processed/X_test_full.csv',  index=False)
print('Full feature sets saved.')


## 4. Mutual Information Feature Selection

Sampling 20% of training data for MI estimation – large enough for stable scores.

In [ ]:
sample_frac = 0.20
X_sample = X_train.sample(frac=sample_frac, random_state=RANDOM_STATE)
y_sample = y_train.loc[X_sample.index]          # Series.loc – safe index alignment

discrete_mask = [col in cat_cols for col in X_train.columns]

print(f'Computing MI on {len(X_sample):,} samples ({sample_frac*100:.0f}% of training data)...')
mi_scores = mutual_info_classif(
    X_sample, y_sample,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE
)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print('\nTop 15 features by Mutual Information:')
print(mi_series.head(15).round(4))

plt.figure(figsize=(10, 7))
mi_series.head(25).plot(kind='barh', color='steelblue')
plt.title('Top 25 Features by Mutual Information Score')
plt.xlabel('MI Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/mi_scores.png', dpi=150)
plt.show()


## 5. Random Forest Feature Importance

**Fix:** increased to 200 estimators and max_depth=15 (matching README) for stable, reliable importance estimates.

In [ ]:
# 15% sample for RF – better stability than 10%
X_sample_rf = X_train.sample(frac=0.15, random_state=RANDOM_STATE)
y_sample_rf  = y_train.loc[X_sample_rf.index]   # Series.loc alignment

# FIX: 200 estimators + max_depth=15 matches README and gives stable importance
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, max_features='sqrt',
    min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE,
    class_weight='balanced'
)
print('Fitting Random Forest on {:,} samples...'.format(len(X_sample_rf)))
rf.fit(X_sample_rf, y_sample_rf)

rf_series = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('\nTop 15 features by RF Importance:')
print(rf_series.head(15).round(4))

plt.figure(figsize=(10, 7))
rf_series.head(25).plot(kind='barh', color='darkorange')
plt.title('Top 25 Features by Random Forest Importance')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/rf_importance.png', dpi=150)
plt.show()


## 6. Consensus Ranking (MI + RF)

In [ ]:
rankings = pd.DataFrame({'MI_Score': mi_series, 'RF_Importance': rf_series})
rankings['MI_Norm']       = rankings['MI_Score']      / rankings['MI_Score'].max()
rankings['RF_Norm']       = rankings['RF_Importance'] / rankings['RF_Importance'].max()
rankings['Combined_Score']= (rankings['MI_Norm'] + rankings['RF_Norm']) / 2
rankings = rankings.sort_values('Combined_Score', ascending=False)

print('Top 15 by Combined Ranking:')
print(rankings['Combined_Score'].head(15).round(4))

plt.figure(figsize=(12, 8))
rankings['Combined_Score'].head(30).plot(kind='barh', color='teal')
plt.title('Top 30 Features – Consensus Ranking (MI + RF)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/combined_importance.png', dpi=150)
plt.show()


## 7. Visualise Top Feature Distributions

In [ ]:
top_10 = rankings.head(10).index.tolist()
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()
sample_idx = X_train.sample(min(5000, len(X_train)), random_state=42).index
for i, col in enumerate(top_10):
    sns.kdeplot(data=X_train.loc[sample_idx], x=col,
                hue=y_train.loc[sample_idx].astype(str),
                ax=axes[i], fill=True, common_norm=False)
    axes[i].set_title(f'{col} by Fraud')
plt.tight_layout()
plt.savefig('../results/figures/top_features_distribution.png', dpi=150)
plt.show()


### Correlation Among Top 20 Selected Features

In [ ]:
top_20 = rankings.head(20).index.tolist()
plt.figure(figsize=(14, 12))
sns.heatmap(X_train[top_20].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.3)
plt.title('Correlation Heatmap – Top 20 Selected Features')
plt.tight_layout()
plt.savefig('../results/figures/top_features_correlation.png', dpi=150)
plt.show()


## 8. Select Top 50 Features & Save

In [ ]:
k = 50
selected_features = rankings.head(k).index.tolist()
print(f'Selected top {k} features (MI + RF consensus).')

X_train_selected = X_train[selected_features]
X_test_selected  = X_test[selected_features]
print('Train selected shape:', X_train_selected.shape)
print('Test  selected shape:', X_test_selected.shape)

# Save feature list
pd.Series(selected_features).to_csv(
    '../data/processed/selected_features.csv', index=False, header=False)

# Save selected datasets
# Note: all features already StandardScaled from 02_preprocessing – no re-scaling needed
X_train_selected.to_parquet('../data/processed/X_train_selected.parquet', index=False)
X_test_selected.to_parquet( '../data/processed/X_test_selected.parquet',  index=False)
X_train_selected.to_csv('../data/processed/X_train_selected.csv', index=False)
X_test_selected.to_csv( '../data/processed/X_test_selected.csv',  index=False)
print('Selected feature sets saved.')


## 9. PCA on Selected Features

Since all features are already uniformly scaled from preprocessing, PCA operates on a clean standardised input – no re-scaling distortion.

In [ ]:
# PCA on already-scaled selected features
# Apply a fresh StandardScaler here purely as a safety measure;
# on already-standardised data this is a near-identity transformation.
pca_scaler    = StandardScaler()
X_train_scaled = pca_scaler.fit_transform(X_train_selected)
X_test_scaled  = pca_scaler.transform(X_test_selected)

pca           = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca   = pca.fit_transform(X_train_scaled)
X_test_pca    = pca.transform(X_test_scaled)

print(f'Input features    : {X_train_selected.shape[1]}')
print(f'PCA components    : {pca.n_components_}  (95% variance)')
print(f'Cumulative variance: {np.cumsum(pca.explained_variance_ratio_)[-1]:.4f}')


### PCA Variance & Loadings

In [ ]:
# Cumulative variance plot
plt.figure(figsize=(9, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', color='purple')
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA – Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../results/figures/pca_variance.png', dpi=150)
plt.show()

# PC1 & PC2 loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=selected_features
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
loadings['PC1'].sort_values(ascending=False).head(15).plot(kind='barh', ax=ax1, color='orange')
ax1.set_title('Top Features → PC1'); ax1.invert_yaxis()
loadings['PC2'].sort_values(ascending=False).head(15).plot(kind='barh', ax=ax2, color='purple')
ax2.set_title('Top Features → PC2'); ax2.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/pca_loadings.png', dpi=150)
plt.show()


## 10. Save PCA Data & Objects

In [ ]:
pc_cols = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]

pd.DataFrame(X_train_pca, columns=pc_cols).to_parquet('../data/processed/X_train_pca.parquet', index=False)
pd.DataFrame(X_test_pca,  columns=pc_cols).to_parquet('../data/processed/X_test_pca.parquet',  index=False)
pd.DataFrame(X_train_pca, columns=pc_cols).to_csv('../data/processed/X_train_pca.csv', index=False)
pd.DataFrame(X_test_pca,  columns=pc_cols).to_csv('../data/processed/X_test_pca.csv',  index=False)

joblib.dump(pca,        '../data/processed/pca.pkl')
joblib.dump(pca_scaler, '../data/processed/pca_scaler.pkl')
print('PCA datasets and objects saved.')
print(f'\nDimensionality reduction summary:')
print(f'  Full (cleaned)  : {X_train.shape[1]} features')
print(f'  Selected (Top50): {X_train_selected.shape[1]} features  ({(1 - 50/X_train.shape[1])*100:.0f}% reduction)')
print(f'  PCA (95% var)   : {X_train_pca.shape[1]} components  ({(1 - X_train_pca.shape[1]/X_train.shape[1])*100:.0f}% reduction)')


## Summary

| Stage | Fix Applied |
|---|---|
| y loading | Loaded as pandas **Series** (not numpy array) for safe `.loc` index alignment |
| RF importance | **200 estimators, max_depth=15** (was 100/10) for stable importance scores |
| Feature scaling | All features already uniformly scaled from 02 – no re-scaling distortion |
| Selected save | X_train_selected saved with same clean scaling as full dataset |
| PCA input | Clean standardised input → correct variance decomposition |

Next: `04_models.ipynb`